# Task 3 — Step 3: GRPO Sequence Normalization Study (Canonical vs. Dr. GRPO)

### Research Question
Recent research (e.g. Dr. GRPO) showed that standard sequence-level averaging ($1/T_k$) can penalize longer responses and bias gradient allocation.
How does changing gradient normalization from **Canonical GRPO** ($1/T_k$) to **Dr. GRPO** ($1/L_{\text{max}}$ token normalization) affect:
1. Response length shift in generated completions?
2. Gradient norm allocation on short vs. long responses?
3. Final held-out quality and reward?


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Normalization Comparison or Load Cached Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task3_grpo.compare_normalization --config configs/grpo.yaml
else:
    print("[OK] Loading saved normalization comparison results...")


## 2. Comparison Metrics Table


In [ ]:
norm_file = "results/task3_grpo/grpo_normalization_comparison.json"
if os.path.exists(norm_file):
    with open(norm_file, encoding='utf-8') as f:
        norm_data = json.load(f)
        
    rows = []
    for mode in ["grpo", "dr_grpo"]:
        ev = norm_data.get(mode, {}).get("eval", {})
        tr = norm_data.get(mode, {}).get("train", {})
        name = "Canonical GRPO (1/T_k)" if mode == "grpo" else "Dr. GRPO (1/L_max)"
        rows.append({
            "Normalization": name,
            "Held-Out Reward": f"{ev.get('mean_reward', 0):.4f}",
            "Mean KL Drift": f"{ev.get('mean_kl', 0):.4f}",
            "Response Length (tokens)": f"{ev.get('mean_response_length_tokens', 0):.1f}",
            "EOS Termination Rate": f"{ev.get('eos_termination_rate', 0)*100:.1f}%"
        })
    df_norm = pd.DataFrame(rows)
    display(df_norm)


## 3. Visualizing Length Shift Between Normalization Methods


In [ ]:
if os.path.exists(norm_file):
    modes = [r["Normalization"] for r in rows]
    rews = [float(r["Held-Out Reward"]) for r in rows]
    lens = [float(r["Response Length (tokens)"]) for r in rows]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Reward
    axes[0].bar(modes, rews, color=['#3B82F6', '#10B981'], width=0.4, alpha=0.85)
    axes[0].set_ylabel('Mean Reward')
    axes[0].set_title('Reward Comparison: Canonical vs. Dr. GRPO')
    axes[0].grid(axis='y', alpha=0.3)
    
    # 2. Length
    axes[1].bar(modes, lens, color=['#3B82F6', '#10B981'], width=0.4, alpha=0.85)
    for i, l in enumerate(lens):
        axes[1].text(i, l + 2, f"{l:.1f}", ha='center', fontweight='bold')
    axes[1].set_ylabel('Mean Response Length (tokens)')
    axes[1].set_title('Generated Length Shift by Normalization Method')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Research Question 2 Analysis
- Did Canonical GRPO produce shorter completions due to $1/T_k$ gradient weighting?
- How did Dr. GRPO alter response verbosity and optimization stability?
